# 03 · Phenology metrics per year and topographic zone

Loads the cached Sentinel-2 statistics, filters dates by quality and derives the start, peak and end of the autumn yellowing season for every year and zone with several methods from the literature. It compares the methods, projects the chosen dates a few years ahead and relates them to elevation and aspect.

| Section | Content |
|---|---|
| 1 | load and filter the cached statistics |
| 2 | the indices over the years |
| 3 | curves fitted per year and zone (the basis of every date method) |
| 4 / 5 / 6 | start / peak / end of the season: every method |
| 7 | comparison of all methods and the chosen date per date type |
| 8 | OLS trend and projection of the chosen dates |
| 9 | chosen dates vs elevation and aspect |
| 10 | STL decomposition of the indices |
| 11 | export |

- **Settings:** `config.py` (years, date filters, fit settings, `SEASON_CHOICE` for sections 8 and 9)
- **Reads:**
  - `cache/<AOI>_zones/` (from `01_download_sentinel2`)
  - `cache/validation/<AOI>/` (from `01b_download_validation`, optional)
- **Writes** to `results/<AOI>/`:
  - `season_dates.csv`: start / mid / peak / end of every method, index, zone and year, with quality columns
  - `season_method_comparison.csv`: the comparison of section 7
  - `season_selected.csv`: the chosen start / peak / end per zone and year
  - `season_projection.csv`: the projection of section 8
  - `zones.csv`, `topography_effects.csv`
  - `eos_by_zone.csv` and `logistic_by_zone.csv`: inputs of `04_climate_analysis`
  - `ndvi_timeseries_all_dates.csv`, `ndvi_timeseries_clean.csv`

Years whose autumn is not fully cached yet (`SEASON_COMPLETE_DOY`, e.g. the current year) are shown as preliminary in the plots but left out of all tables, trends and models.

Runs offline in one to a few minutes (most of it the bootstrap, `N_BOOT`). Sources are listed at the end.

## Yellowing season: start, peak and end

The aim is three dates per year and zone:

| Date | Meaning | Signal | Methods (sections 4–6) | Checked against (section 7) |
|---|---|---|---|---|
| **Start** | needles begin to lose chlorophyll | greenness (GRVI, GCC, NDVI) leaves its summer level | `start`: 90 % of the autumn amplitude left | order, data support |
| **Peak** | canopy most yellow: chlorophyll gone, needles still on | yellowness (RCC, NDYI, PSRI) at its maximum; greenness half-way down | `peak`: maximum of a yellowness index; `mid`: 50 % of the greenness decline | MeteoSwiss needle colouring 50 % |
| **End** | needles have fallen | NDVI reaches its post-season level | `end`: 15 % of the amplitude left | MeteoSwiss needle drop 50 %, HR-VPP end of season |

- **Why greenness for the start and NDVI for the end:** the change from green to yellow shows mainly in the visible bands (green falls relative to red; Motohka et al. 2010). Yellow needles still reflect NIR strongly, so NDVI drops mostly with needle fall.
- **Thresholds:** 90 / 50 / 15 % of the amplitude follow MODIS MCD12Q2 (senescence / greendown midpoint / dormancy; Gray et al. 2019) and HR-VPP (end of season at 15 %; Tian et al. 2021).
- **Expected:** start < peak < end in every year and zone; earlier at higher elevation.
- **Limits:** after mid-October there are few clear observations (low sun turns north slopes SCL "dark", clouds, early snow). Every date therefore carries quality columns (section 3g); the figures of sections 4–6 show dates inside a data gap hollow.

In [ ]:
from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings

from matplotlib.colors import LightSource
from scipy import sparse
from scipy.optimize import OptimizeWarning, curve_fit
from scipy.sparse.linalg import spsolve
from statsmodels.tsa.seasonal import STL

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import clean_frame, incomplete_years, index_frame, load_stats, load_zone_table

INDEX_COLORS = {"NDVI": "darkgreen", "GRVI": "darkorange", "PSRI": "purple", "GCC": "yellowgreen",
                "RCC": "firebrick", "NDYI": "goldenrod"}
INDEX_MARKERS = {"NDVI": "o", "GRVI": "s", "PSRI": "^", "GCC": "D", "RCC": "v", "NDYI": "P"}
ASPECT_CODES = {"N": 0, "S": 1, "flat": 2}   # aspect classes as stored in zones.tif (01)
ASPECT_COLORS = {"N": "tab:blue", "S": "tab:orange", "flat": "grey"}
METHOD_COLORS = {"MA midpoint": "tab:purple", "MA fixed threshold": "plum", "logistic, free": "tab:red",
                 "logistic, fixed base": "tab:blue", "double logistic": "tab:orange",
                 "double logistic, 3rd derivative": "tab:brown", "Whittaker": "tab:green"}
print(f"AOI: {AOI_NAME} | years {FIRST_YEAR}-{LAST_YEAR} | results: {OUT_DIR}")

## 1. Load and filter the cached statistics

A date is kept for a zone if at least `MIN_VALID_FRACTION` of the zone's pixels are clear (and it passes `MAX_SCENE_CLOUD` / `MAX_SCL_FRACTION`). With `VALID_SHARE_OF = "covered"`, this and the other SCL shares count only the zone pixels inside the scene's footprint: every second scene comes from the edge of a neighbouring orbit's swath and covers only part of the AOI, and such a scene now passes if its covered part is clear and holds at least `MIN_COVERED_PIXELS` pixels. `df`, `df_clean` and so on refer to the whole AOI (zone `"all"`). `clean_by_zone[index][zone]` holds the cleaned median series of every zone. `INCOMPLETE` lists the years whose cache ends before `SEASON_COMPLETE_DOY`. Indices of `INDICES` that the cache lacks (e.g. GCC / RCC / NDYI before `01` was re-run) are skipped.

In [ ]:
stats_all = load_stats()
zone_table = load_zone_table()
ZONES = list(zone_table.index)   # "all" first, then elevation band x aspect
INCOMPLETE = incomplete_years(stats_all)
if INCOMPLETE:
    print(f"Autumn not fully cached (preliminary, left out of tables and models): {INCOMPLETE}")

missing = [n for n in INDICES if f"{n.lower()}_median" not in stats_all or stats_all[f"{n.lower()}_median"].isna().all()]
if missing:
    print(f"Not in the cache, skipped: {missing} (re-run 01_download_sentinel2 to add them)")
INDICES = [n for n in INDICES if n not in missing]

frames = {name: index_frame(stats_all, name.lower()) for name in INDICES}
clean = {name: clean_frame(stats_all, name.lower()) for name in INDICES}
df, df_clean = frames["NDVI"], clean["NDVI"]
for name in INDICES:
    print(f"{name}: {len(clean[name])} of {len(frames[name])} dates kept")

clean_by_zone = {name: {zone: clean_frame(stats_all, name.lower(), zone)[f"{name.lower()}_median"] for zone in ZONES}
                 for name in INDICES}
display(zone_table[["n_pixels", "elev_mean", "slope_mean", "northness_mean"]].round(2).join(
    pd.DataFrame({name: {z: len(s) for z, s in by_zone.items()} for name, by_zone in clean_by_zone.items()})
      .add_prefix("dates_")))

### 1b. Check: partial-swath scenes vs full scenes

A partial scene always sees the same strip of a zone, from a different viewing angle. If that strip is not representative, its zone median is offset from the full-scene medians. Each kept partial scene (covering < 90 % of the zone) is paired with the nearest kept full scene at most `PAIR_MAX_DAYS` days away (usually 2), both without snow.

- **`bias`:** mean difference partial − full; near 0 = no systematic offset
- **`mad`:** median absolute difference, including real change over the days in between
- **`full_mad`:** the same for pairs of full scenes ≤ 5 days apart, as the noise to compare against
- **`bias_share`:** `bias` relative to the zone's seasonal range (5th–95th percentile)

A `bias_share` of a few percent shifts a transition date by much less than a day. If it is large for a zone, set `VALID_SHARE_OF = "zone"` or raise `MIN_COVERED_PIXELS`.

In [ ]:
PAIR_MAX_DAYS = 3
PARTIAL_COVER = 0.9   # scenes covering less of the zone count as partial


def paired_diff(a, b, max_days):
    # a - b for each date of a and the nearest other date of b at most max_days away (series indexed by date)
    m = pd.merge_asof(a.rename("a").to_frame(), b.rename("b").to_frame(), left_index=True, right_index=True,
                      direction="nearest", tolerance=pd.Timedelta(days=max_days), allow_exact_matches=False)
    return (m.a - m.b).dropna()


rows = []
for name in INDICES:
    col = f"{name.lower()}_median"
    for zone in ZONES:
        f = clean_frame(stats_all, name.lower(), zone)
        f = f[f.scl_snow_fraction <= MAX_SNOW_FRACTION]
        partial = f.covered_fraction < PARTIAL_COVER
        diff = paired_diff(f.loc[partial, col], f.loc[~partial, col], PAIR_MAX_DAYS)
        full_diff = paired_diff(f.loc[~partial, col], f.loc[~partial, col], 5)
        spread = f[col].quantile(0.95) - f[col].quantile(0.05)
        rows.append({"index": name, "zone": zone, "partial_kept": int(partial.sum()), "n_pairs": len(diff),
                     "bias": diff.mean(), "mad": diff.abs().median(), "full_mad": full_diff.abs().median(),
                     "bias_share": diff.mean() / spread})
partial_check = pd.DataFrame(rows).set_index(["index", "zone"])
print(f"VALID_SHARE_OF = {VALID_SHARE_OF!r}, MIN_COVERED_PIXELS = {MIN_COVERED_PIXELS}")
display(partial_check.round(3))

## 2. The indices over the years (whole AOI)

- **Top:** every index over the whole period: all dates (grey), the dates that pass the filter (colour) and a centred moving average (`SMOOTH_WINDOW` × 5 days) on a regular 5-day grid, with gaps of up to 30 days interpolated. Shaded: preliminary years.
- **Bottom:** the moving average of each year over the snow-free season (`SEASON_WINDOW`), one line per year.

The moving average is only for display and for the two baseline methods `MA midpoint` and `MA fixed threshold` (3e); all other methods work on the observations directly.

In [ ]:
def smooth(series):
    # Centred moving average on a regular 5-day grid; gaps of up to 30 days interpolated, longer ones left empty
    regular = series.resample("5D").mean().interpolate(method="time", limit=6)
    rolled = regular.rolling(SMOOTH_WINDOW, center=True, min_periods=3).mean()
    return rolled.where(regular.notna())


def clear_limits(name, margin=0.15):
    # y limits from the clear dates (snow and cloud values of the unfiltered dates would squash the plot)
    lo, hi = medians[name].quantile([0.005, 0.995])
    return lo - margin * (hi - lo), hi + margin * (hi - lo)


medians = {name: clean[name][f"{name.lower()}_median"] for name in INDICES}
smoothed = {name: smooth(s) for name, s in medians.items()}

fig, axes = plt.subplots(len(INDICES), 1, figsize=(13, 2.4 * len(INDICES)), sharex=True, squeeze=False)
for ax, name in zip(axes[:, 0], INDICES):
    ax.scatter(frames[name].index, frames[name][f"{name.lower()}_median"], s=5, color="lightgrey", label="all dates")
    ax.scatter(medians[name].index, medians[name], s=8, color=INDEX_COLORS[name], label="clear dates")
    ax.plot(smoothed[name].index, smoothed[name], color="black", lw=1.2,
            label=f"{SMOOTH_WINDOW * 5}-day moving average")
    for year in INCOMPLETE:
        ax.axvspan(pd.Timestamp(year, 1, 1), pd.Timestamp(year, 12, 31), color="grey", alpha=0.1, lw=0)
    ax.set_ylim(*clear_limits(name))
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
axes[0, 0].legend(loc="lower left", ncol=3, fontsize=8)
fig.suptitle("Indices over the years (whole AOI)")
plt.tight_layout()
plt.show()

years = sorted(set(stats_all.index.year))
year_colors = dict(zip(years, plt.cm.viridis(np.linspace(0, 1, len(years)))))
fig, axes = plt.subplots(1, len(INDICES), figsize=(4.5 * len(INDICES), 3.8), squeeze=False)
for ax, name in zip(axes[0], INDICES):
    s = smoothed[name]
    for year, color in year_colors.items():
        y = s[s.index.year == year]
        ax.plot(y.index.dayofyear, y, color=color, lw=1.2, ls="--" if year in INCOMPLETE else "-", label=str(year))
    ax.set_xlim(*SEASON_WINDOW)
    ax.set_ylim(*clear_limits(name))
    ax.set_title(name)
    ax.set_xlabel("day of year")
    ax.grid(alpha=0.3)
axes[0, -1].legend(fontsize=7, ncol=2)
fig.suptitle(f"Seasonal course per year ({SMOOTH_WINDOW * 5}-day moving average, whole AOI; dashed: preliminary)")
plt.tight_layout()
plt.show()

## 3. Curves fitted per year and zone

Every date method of sections 4–6 reads its dates off one of these curves, fitted to the clear observations of each index, zone and year:

| Curve | Fit | Source |
|---|---|---|
| `Whittaker` | weighted Whittaker smoother, daily, over the whole series (3b) | Eilers 2003; Atzberger & Eilers 2011 |
| `logistic, free` | autumn logistic, both levels free, unweighted (3c) | earlier version of this notebook |
| `logistic, fixed base` | autumn logistic, post-season level fixed, weighted, bootstrap CIs (3d) | Beck et al. 2006 (fixed base) |
| `double logistic` | whole snow-free season, base fixed, weighted (3d) | Beck et al. 2006; TIMESAT / HR-VPP (Jönsson & Eklundh 2004) |
| moving average (`MA ...`) | NDVI only, the curve of section 2 (3e) | baseline |

All dates go into one table, `season_dates` (3g); 3h shows the curves of the whole AOI. Overview of the methods: Zeng et al. 2020.

### 3a. Observations, weights, outliers and the post-season level

- **Weights** follow the HR-VPP quality classes: 1 for a good date (clear share ≥ `GOOD_VALID_FRACTION`), 0.5 for a usable one, 0 for a date with snow on more than `MAX_SNOW_FRACTION` of the zone.
- **Outliers** follow the HR-VPP rule: a date is dropped (weight 0) if its residual from a stiff smoothing curve exceeds `OUTLIER_FACTOR` × the global and × the local (91-day) median residual.
- **Post-season level ("base"):** the snow-free level after needle fall, the median of the clear dates in `POST_WINDOW` over all years. It is held fixed in the fits instead of being fitted per year, as Beck et al. (2006) do with the winter NDVI. Without a fixed base, a fit with no data after the decline puts the lower level anywhere (the `at_bound` fits of 3c). Zones with fewer than `MIN_BASE_OBS` such dates use the whole AOI's level.

In [ ]:
def obs_frame(name, zone):
    # Clear observations of one index and zone: y (zone median), w (weight), year, doy. Weights as the HR-VPP
    # quality classes (ATBD; Tian et al. 2021): 1 = good, 0.5 = usable, 0 = snow (kept for the plots, not fitted)
    f = clean_frame(stats_all, name.lower(), zone)
    obs = pd.DataFrame({"y": f[f"{name.lower()}_median"],
                        "w": np.where(f.valid_fraction >= GOOD_VALID_FRACTION, 1.0, 0.5)}, index=f.index)
    obs.loc[f.scl_snow_fraction > MAX_SNOW_FRACTION, "w"] = 0.0
    obs = obs.dropna(subset=["y"])
    return obs.assign(year=obs.index.year, doy=obs.index.dayofyear, outlier=False)


def base_levels(name):
    # Post-season level per zone, held fixed in all fits (Beck et al. 2006): median of the weighted dates in
    # POST_WINDOW over all years. Fallbacks: the whole AOI's level; then, as HR-VPP sets its base level, the 5th
    # percentile of all clear dates (95th for rising indices, whose post-season level is the high one)
    rows = {}
    for zone in ZONES:
        o = obs_by_zone[name][zone]
        post = o[o.doy.between(*POST_WINDOW) & (o.w > 0)]
        rows[zone] = {"level": post.y.median(), "n_dates": len(post), "source": "zone"}
    for zone, r in rows.items():
        if r["n_dates"] < MIN_BASE_OBS and rows["all"]["n_dates"] >= MIN_BASE_OBS:
            r.update(level=rows["all"]["level"], source="whole AOI")
        elif r["n_dates"] < MIN_BASE_OBS:
            q = 0.05 if EXPECT_DECREASE[name] else 0.95
            r.update(level=obs_by_zone[name][zone].y.quantile(q), source=f"{q:.0%} percentile")
    return pd.DataFrame(rows).T.astype({"level": float, "n_dates": int})


# ---- Weighted Whittaker smoother (also used for the outlier test, so it is defined here)
def whittaker(days, y, w, lam, n):
    # Eilers (2003), for vegetation indices Atzberger & Eilers (2011): on a daily grid of n days, minimise
    # sum w (y - z)^2 + lam * sum (2nd difference of z)^2. Days without an observation have weight 0, so irregular
    # dates and gaps need no interpolation beforehand; across a gap the curve becomes nearly a straight line.
    W, Y = np.zeros(n), np.zeros(n)
    np.add.at(W, days, w)
    np.add.at(Y, days, w * y)
    D = sparse.diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n - 2, n))
    return spsolve((sparse.diags(W) + lam * (D.T @ D)).tocsc(), Y)


def smooth_series(obs, lam, base):
    # Daily Whittaker curve (pd.Series over whole calendar years) through the weighted dates of obs. Long
    # off-season gaps are filled as in HR-VPP: if more than 90 days pass without a good date between the autumn
    # and the spring equinox, control points at the base level (weight 0.5) are added every 15 days.
    o = obs[obs.w > 0]
    start = pd.Timestamp(obs.index.min().year, 1, 1)
    end = pd.Timestamp(obs.index.max().year, 12, 31)
    grid = pd.date_range(start, end)
    days, y, w = (o.index - start).days.values, o.y.values, o.w.values
    control = [d for a, b in zip(days[:-1], days[1:]) if b - a > 90 for d in range(a + 15, b - 7, 15)
               if not 80 < grid[d].dayofyear < 266]
    days = np.concatenate([days, control]).astype(int)
    y = np.concatenate([y, np.full(len(control), base)])
    w = np.concatenate([w, np.full(len(control), 0.5)])
    return pd.Series(whittaker(days, y, w, lam, len(grid)), index=grid)


def flag_outliers(obs, base):
    # HR-VPP outlier rule (ATBD 4.2.1): residuals from a stiff curve; outlier if the residual exceeds
    # OUTLIER_FACTOR x the global median residual and OUTLIER_FACTOR x the median of the surrounding 91 days
    o = obs[obs.w > 0]
    curve = smooth_series(obs, max(WHITTAKER_LAMBDAS), base)
    res = (o.y - curve.reindex(o.index).values).abs()
    local = res.rolling("91D", center=True, min_periods=1).median()
    out = res.index[(res > OUTLIER_FACTOR * res.median()) & (res > OUTLIER_FACTOR * local)]
    obs = obs.copy()
    obs.loc[out, ["w", "outlier"]] = [0.0, True]
    return obs


obs_by_zone = {name: {zone: obs_frame(name, zone) for zone in ZONES} for name in INDICES}
base = {name: base_levels(name) for name in INDICES}
for name in INDICES:
    for zone in ZONES:
        obs_by_zone[name][zone] = flag_outliers(obs_by_zone[name][zone], base[name].level[zone])

display(pd.concat({name: b for name, b in base.items()}, axis=1).round(3))
print("Dates per index and zone: weight 1 / 0.5 / 0 (snow) / outliers")
display(pd.DataFrame({name: {zone: f"{(o.w == 1).sum()} / {(o.w == 0.5).sum()} / {((o.w == 0) & ~o.outlier).sum()} / "
                                   f"{o.outlier.sum()}" for zone, o in by_zone.items()}
                      for name, by_zone in obs_by_zone.items()}))

### 3b. Whittaker smoother

The weighted Whittaker smoother (Eilers 2003) fits a smooth daily curve to the irregular observations; Atzberger & Eilers (2011) introduced it for vegetation-index time series. It needs no gap interpolation beforehand, and one parameter, λ, sets the smoothness. λ is chosen per index from `WHITTAKER_LAMBDAS` by 10-fold cross-validation on the whole AOI (weighted RMSE of left-out dates), then used for every zone. HR-VPP assumes clear Sentinel-2 dates are unbiased and applies no upper-envelope correction; neither does this notebook.

In [ ]:
def choose_lambda(obs, base_level, k=10, seed=0):
    # Weighted RMSE of left-out good dates for every lam (k-fold cross-validation)
    good = obs[obs.w > 0]
    fold = np.random.default_rng(seed).integers(0, k, len(good))
    rmse = {}
    for lam in WHITTAKER_LAMBDAS:
        sq = []
        for i in range(k):
            test = good[fold == i]
            curve = smooth_series(obs.drop(index=test.index), lam, base_level)
            sq.append(test.w * (test.y - curve.reindex(test.index).values) ** 2)
        rmse[lam] = np.sqrt(pd.concat(sq).sum() / good.w.sum())
    return pd.Series(rmse)


cv_lambda = pd.DataFrame({name: choose_lambda(obs_by_zone[name]["all"], base[name].level["all"]) for name in INDICES})
LAMBDA = cv_lambda.idxmin()
print("Cross-validated RMSE per lambda (whole AOI); chosen:", LAMBDA.to_dict())
display(cv_lambda.rename_axis("lambda").round(4))
wcurves = {name: {zone: smooth_series(obs_by_zone[name][zone], LAMBDA[name], base[name].level[zone]) for zone in ZONES}
           for name in INDICES}

### 3c. Logistic fit with free levels (raw clear observations)

For each year, index and zone, a logistic curve is fitted to the raw clear observations within `FIT_WINDOW` (day of year):

`y(t) = lo + (hi - lo) / (1 + exp((t - t0) / s))`

- `hi` is the summer level, `lo` the post-transition level, `t0` the 50 % point and `s` the time scale in days.
- NDVI and GRVI drop (`hi > lo`); PSRI rises (`hi < lo`).
- **onset / mid / end** = 20 / 50 / 80 % of the transition: `t = t0 + s · ln(f / (1 − f))`.

Quality columns:
- `n_obs`: clear observations used in the fit
- `n_before_mid` / `n_after_mid`: observations on each side of `t0`
- `rmse`: fit error
- `at_bound`: a parameter sits on its fit bound. Mostly `scale_days` = 1, a step at the edge of a data gap rather than a measured transition, or `lo` = ±2, a level not constrained by any observation
- `direction_ok`: the transition goes the expected way for that index

`logistic_table` covers complete years; `logistic_table_all` also has the preliminary years. `04_climate_analysis` reads it (`logistic_by_zone.csv`).

In [ ]:
def logistic(t, hi, lo, t0, s):
    return lo + (hi - lo) / (1 + np.exp((t - t0) / s))


LOGISTIC_BOUNDS = ([-2, -2, FIT_WINDOW[0], 1], [2, 2, FIT_WINDOW[1], 40])   # hi, lo, t0, s


def fit_logistic(doy, y):
    split = np.median(doy)
    p0 = [np.median(y[doy <= split]), np.median(y[doy > split]), 280, 7]
    popt, _ = curve_fit(logistic, doy, y, p0=p0, bounds=LOGISTIC_BOUNDS, maxfev=20000)
    return popt


def at_bound(popt):
    return bool(any(np.isclose(popt, b, rtol=0, atol=1e-3).any() for b in LOGISTIC_BOUNDS))


def logistic_rows(name, series, zone="all"):
    # Per-year fit of one index's clear observations; returns the table rows and {year: (doy, y, popt)}
    rows, fits = [], {}
    ser = series.dropna()
    for year, s in ser.groupby(ser.index.year):
        doy = s.index.dayofyear.values.astype(float)
        sel = (doy >= FIT_WINDOW[0]) & (doy <= FIT_WINDOW[1])
        doy, y = doy[sel], s.values[sel]
        if len(y) == 0:
            continue
        row = {"index": name, "zone": zone, "year": year, "n_obs": len(y)}
        popt = None
        if len(y) >= MIN_OBS_PER_YEAR:
            try:
                popt = fit_logistic(doy, y)
            except RuntimeError:
                print(f"{name} {zone} {year}: fit did not converge")
        if popt is not None:
            hi, lo, t0, sc = popt
            row.update({
                "hi": hi, "lo": lo, "scale_days": sc,
                "rmse": np.sqrt(np.mean((logistic(doy, *popt) - y) ** 2)),
                "n_before_mid": int((doy < t0).sum()),
                "n_after_mid": int((doy >= t0).sum()),
                "direction_ok": bool((hi > lo) == EXPECT_DECREASE[name]),
                "at_bound": at_bound(popt),
            })
            for label, f in TRANSITION_FRACTIONS.items():
                d = t0 + sc * np.log(f / (1 - f))
                row[f"{label}_doy"] = d
                row[f"{label}_date"] = date(int(year), 1, 1) + timedelta(days=int(round(d)) - 1)
        fits[year] = (doy, y, popt)
        rows.append(row)
    return rows, fits


rows, fits = [], {}
for name in INDICES:
    for zone in ZONES:
        zone_rows, zone_fits = logistic_rows(name, clean_by_zone[name][zone], zone)
        rows += zone_rows
        if zone == "all":
            fits.update({(name, year): fit for year, fit in zone_fits.items()})
logistic_table_all = pd.DataFrame(rows).set_index(["index", "zone", "year"]).round(3)
logistic_table = logistic_table_all.drop(index=INCOMPLETE, level="year", errors="ignore")
n_fit = logistic_table_all.hi.notna().sum()
print(f"{n_fit} free logistic fits, {int(logistic_table_all.at_bound.eq(True).sum())} with a parameter on its bound")

### 3d. Logistic fits with a fixed post-season level

- **`logistic, fixed base`:** the autumn logistic of 3c, `y = lo + (hi − lo) / (1 + exp((t − t0) / s))`, but with `lo` fixed at the post-season level (3a) and the dates weighted. Only `hi`, `t0` and `s` are fitted, to the weighted dates in `FIT_WINDOW[0]`–`POST_WINDOW[1]`.
- **`double logistic`:** the function of Beck et al. (2006) over the whole snow-free season (`SEASON_WINDOW`), `y = b + (m − b)·(1/(1 + exp(−mS(t − S))) + 1/(1 + exp(mA(t − A))) − 1)`, with the base `b` fixed. It is the function type TIMESAT fits for HR-VPP; only its autumn half (`A`, `mA`) gives dates here.
- **Dates:** a share `f` of the amplitude is left at `t = t0 + s · ln((1 − f) / f)` (for the double logistic `t0 = A`, `s = 1 / mA`).
- **`double logistic, 3rd derivative`:** Busetto et al. (2010) found, for larch in the Aosta Valley, that the zeros of the third derivative of a double-logistic NDVI curve matched field observations best (MAE 4 days for the end). For a logistic they lie at `t0 ∓ s · ln(2 + √3)`, where about 79 % and 21 % of the amplitude are left.
- **Uncertainty:** the fixed-base logistic is refitted to `N_BOOT` bootstrap resamples of each year's dates; `ci_low` / `ci_high` are the 2.5 / 97.5 % quantiles of each date.
- **`at_bound`:** `t0` (or `A`) or the time scale sits on its bound.

In [ ]:
TURNING = np.log(2 + np.sqrt(3))   # 3rd-derivative zeros of a logistic: t0 -/+ 1.317 s (Busetto et al. 2010)
QA_LEVELS = ["filled", "low", "medium", "high"]


def logistic_base(t, hi, t0, s, lo):
    return lo + (hi - lo) / (1 + np.exp((t - t0) / s))


def double_logistic(t, m, S, mS, A, mA, b):
    # Beck et al. (2006): b = off-season level (fixed), m = summer level, S / A = spring / autumn inflection day,
    # mS / mA = their rates (1/day)
    return b + (m - b) * (1 / (1 + np.exp(-mS * (t - S))) + 1 / (1 + np.exp(mA * (t - A))) - 1)


def fraction_day(t0, s, f):
    # Day on a logistic when a share f of the amplitude is left
    return t0 + s * np.log((1 - f) / f)


def fit_fixed_base(doy, y, w, lo):
    p0 = [np.clip(np.median(y[doy <= np.median(doy)]), -1.9, 1.9), 280, 7]
    bounds = ([-2, FIT_WINDOW[0], 1], [2, POST_WINDOW[1], 40])
    popt, _ = curve_fit(lambda t, hi, t0, s: logistic_base(t, hi, t0, s, lo), doy, y, p0=p0,
                        sigma=1 / np.sqrt(w), bounds=bounds, maxfev=20000)
    return popt


def fit_double_logistic(doy, y, w, b):
    summer = y[(doy >= SUMMER_WINDOW[0]) & (doy <= SUMMER_WINDOW[1])]
    p0 = [np.clip(np.median(summer if len(summer) else y), -1.9, 1.9), 160, 0.1, 280, 0.1]
    bounds = ([-2, SEASON_WINDOW[0], 1 / 40, SUMMER_WINDOW[0], 1 / 40], [2, SUMMER_WINDOW[0], 1, SEASON_WINDOW[1], 1])
    popt, _ = curve_fit(lambda t, m, S, mS, A, mA: double_logistic(t, m, S, mS, A, mA, b), doy, y, p0=p0,
                        sigma=1 / np.sqrt(w), bounds=bounds, maxfev=20000)
    return popt


def hrvpp_qa(doy, t80, t20):
    # Adapted from the HR-VPP QA (ATBD Table 3): clear dates are counted in three parts of the curve - plateau
    # (30 days before 80 % of the amplitude is left), decline (80 % -> 20 %) and after (30 days past 20 %).
    # high = more than 2 dates in all three parts, medium = in two, low = in one, filled = in none
    parts = [(doy >= t80 - 30) & (doy < t80), (doy >= t80) & (doy <= t20), (doy > t20) & (doy <= t20 + 30)]
    return QA_LEVELS[sum(int(p.sum()) > 2 for p in parts)]


def on_bound(value, low, high):
    return bool(np.isclose(value, low, atol=1e-3) or np.isclose(value, high, atol=1e-3))


rng = np.random.default_rng(0)
rows, fixed_fits, dl_fits = [], {}, {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", OptimizeWarning)
    for name in INDICES:
        for zone in ZONES:
            obs, lo = obs_by_zone[name][zone], base[name].level[zone]
            for year, o in obs[obs.w > 0].groupby("year"):
                key = dict(index=name, zone=zone, year=year)
                autumn = o[o.doy.between(FIT_WINDOW[0], POST_WINDOW[1])]
                if len(autumn) < MIN_OBS_PER_YEAR:
                    continue
                doy, y, w = autumn.doy.values.astype(float), autumn.y.values, autumn.w.values
                # -- fixed-base logistic + bootstrap
                try:
                    hi, t0, s = fit_fixed_base(doy, y, w, lo)
                except RuntimeError:
                    hi = None
                if hi is not None:
                    fixed_fits[(name, zone, year)] = (hi, t0, s, lo)
                    boot = []
                    for _ in range(N_BOOT):
                        i = rng.integers(0, len(doy), len(doy))
                        try:
                            b = fit_fixed_base(doy[i], y[i], w[i], lo)
                        except (RuntimeError, ValueError):
                            continue
                        boot.append([fraction_day(b[1], b[2], f) for f in AMPLITUDE_FRACTIONS.values()])
                    boot = np.array(boot)
                    qa = hrvpp_qa(doy, fraction_day(t0, s, 0.8), fraction_day(t0, s, 0.2))
                    bound = on_bound(t0, FIT_WINDOW[0], POST_WINDOW[1]) or on_bound(s, 1, 40)
                    for k, (metric, f) in enumerate(AMPLITUDE_FRACTIONS.items()):
                        ci = (np.percentile(boot[:, k], [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
                              if len(boot) >= 20 else [np.nan, np.nan])
                        rows.append({**key, "method": "logistic, fixed base", "metric": metric,
                                     "doy": fraction_day(t0, s, f), "ci_low": ci[0], "ci_high": ci[1], "qa": qa,
                                     "at_bound": bound, "n_obs": len(doy)})
                # -- double logistic over the season
                season = o[o.doy.between(*SEASON_WINDOW)]
                try:
                    m, S, mS, A, mA = fit_double_logistic(season.doy.values.astype(float), season.y.values,
                                                          season.w.values, lo)
                except RuntimeError:
                    continue
                dl_fits[(name, zone, year)] = (m, S, mS, A, mA, lo)
                qa = hrvpp_qa(doy, fraction_day(A, 1 / mA, 0.8), fraction_day(A, 1 / mA, 0.2))
                bound = on_bound(A, SUMMER_WINDOW[0], SEASON_WINDOW[1]) or on_bound(mA, 1 / 40, 1)
                common = {**key, "qa": qa, "at_bound": bound, "n_obs": len(season)}
                for metric, f in AMPLITUDE_FRACTIONS.items():
                    rows.append({**common, "method": "double logistic", "metric": metric,
                                 "doy": fraction_day(A, 1 / mA, f)})
                for metric, sign in [("start", -1), ("end", 1)]:
                    rows.append({**common, "method": "double logistic, 3rd derivative", "metric": metric,
                                 "doy": A + sign * TURNING / mA})
fit_rows = rows
print(f"{len(fixed_fits)} fixed-base logistic and {len(dl_fits)} double-logistic fits")

### 3e. Moving-average baseline (NDVI)

The first version of this notebook took the end of season from the NDVI moving average of section 2: the first date after the year's maximum at which it drops to
- **`MA fixed threshold`:** `EOS_FIXED_THRESHOLD`, or
- **`MA midpoint`:** halfway between the maximum and the minimum after it.

Years with fewer than `MIN_POINTS_PER_YEAR` values, or without a crossing, get no date. Both are kept as a baseline: the midpoint threshold depends on the last clear value of the year, and the fixed threshold is rarely reached. `eos_table` (`eos_by_zone.csv`) is read by `04_climate_analysis`.

In [ ]:
def first_crossing(post_peak, threshold):
    below = post_peak[post_peak <= threshold]
    return below.index.min() if not below.empty else pd.NaT


def eos_rows(series):
    # One EOS row per year of a smoothed NDVI series
    rows = []
    s_all = series.dropna().sort_index()
    for year, s in s_all.groupby(s_all.index.year):
        if len(s) < MIN_POINTS_PER_YEAR:
            continue
        peak = s.idxmax()
        post_peak = s[s.index >= peak]
        peak_value = s.loc[peak]
        midpoint = post_peak.min() + 0.5 * (peak_value - post_peak.min())
        eos_fixed = first_crossing(post_peak, EOS_FIXED_THRESHOLD)
        eos_mid = first_crossing(post_peak, midpoint)
        rows.append({
            "year": year,
            "n_points": len(s),
            "peak_date": peak.date(),
            "peak_ndvi": round(peak_value, 3),
            "midpoint_threshold": round(midpoint, 3),
            "eos_fixed_date": eos_fixed.date() if pd.notna(eos_fixed) else None,
            "eos_fixed_doy": eos_fixed.dayofyear if pd.notna(eos_fixed) else np.nan,
            "eos_mid_date": eos_mid.date() if pd.notna(eos_mid) else None,
            "eos_mid_doy": eos_mid.dayofyear if pd.notna(eos_mid) else np.nan,
        })
    return rows


rows = [{"zone": zone, **row} for zone in ZONES for row in eos_rows(smooth(clean_by_zone["NDVI"][zone]))]
eos_table_all = pd.DataFrame(rows).set_index(["zone", "year"])
eos_table = eos_table_all.drop(index=INCOMPLETE, level="year", errors="ignore")
print(f"Moving-average dates for {eos_table_all.eos_mid_doy.notna().sum()} (midpoint) and "
      f"{eos_table_all.eos_fixed_doy.notna().sum()} (fixed threshold) of {len(eos_table_all)} zone-years")

### 3f. Dates from the Whittaker curve and the yellowing peak

- **Start / mid / end:** per year, the summer level `hi` is the curve's median in `SUMMER_WINDOW`; the base is the post-season level. With `r` = share of the amplitude left:
  - mid = first day after the summer window with `r ≤ 0.5`
  - start = last day before mid with `r ≥ 0.9` (MCD12Q2: "last date above 90 %")
  - end = first day after mid with `r ≤ 0.15`

  A date that is not reached within the year stays empty.
- **Peak of yellowing:** the day of the curve's maximum in `PEAK_WINDOW` for the yellowness indices of `PEAK_INDICES` (RCC and NDYI, once downloaded, and PSRI). PhenoCam studies track autumn colour with the chromatic coordinates (Sonnentag et al. 2012; Richardson et al. 2018). NDYI was built for yellow canopies (Sulik & Long 2016). PSRI rises with senescence (Merzlyak et al. 1999), but litter and soil keep it high after needle fall, so its maximum may come late. `at_bound` = maximum on the edge of the window. The QA counts clear dates within ±10 days: high ≥ 3, medium 2, low 1, filled 0.

In [ ]:
def whittaker_dates(curve, hi, lo):
    # Start / mid / end on one year of a daily curve (index = DOY); also the 80 % / 20 % days for the QA
    r = (curve - lo) / (hi - lo)
    below_mid = r[(r.index > SUMMER_WINDOW[1]) & (r <= AMPLITUDE_FRACTIONS["mid"])]
    if below_mid.empty:
        return None
    mid = below_mid.index[0]

    def last_above(f):
        above = r[(r.index < mid) & (r >= f)]
        return above.index[-1] if not above.empty else np.nan

    def first_below(f):
        below = r[(r.index > mid) & (r <= f)]
        return below.index[0] if not below.empty else np.nan

    days = {"start": last_above(AMPLITUDE_FRACTIONS["start"]), "mid": mid, "end": first_below(AMPLITUDE_FRACTIONS["end"])}
    return days, last_above(0.8), first_below(0.2)


rows = []
for name in INDICES:
    for zone in ZONES:
        obs, lo = obs_by_zone[name][zone], base[name].level[zone]
        curve_all = wcurves[name][zone]
        for year in sorted(obs.year.unique()):
            curve = curve_all[curve_all.index.year == year]
            curve.index = curve.index.dayofyear
            doy = obs[(obs.year == year) & (obs.w > 0)].doy.values
            key = dict(index=name, zone=zone, year=year, method="Whittaker")
            hi = curve.loc[SUMMER_WINDOW[0]:SUMMER_WINDOW[1]].median()
            found = whittaker_dates(curve, hi, lo) if (hi > lo) == EXPECT_DECREASE[name] else None
            if found:
                days, t80, t20 = found
                qa = hrvpp_qa(doy, t80, t20) if np.isfinite([t80, t20]).all() else "filled"
                rows += [{**key, "metric": m, "doy": d, "qa": qa, "at_bound": False, "n_obs": len(doy)}
                         for m, d in days.items()]
            if name in PEAK_INDICES:
                window = curve.loc[PEAK_WINDOW[0]:PEAK_WINDOW[1]]
                peak = window.idxmax()
                near = int((np.abs(doy - peak) <= 10).sum())
                rows.append({**key, "metric": "peak", "doy": peak, "qa": QA_LEVELS[min(near, 3)],
                             "at_bound": peak in (window.index[0], window.index[-1]), "n_obs": len(doy)})
whittaker_rows = rows
print(f"{len(whittaker_rows)} Whittaker dates")

### 3g. All dates in one table

`season_dates` holds every date of every method, index, zone and year (long format), including the moving-average (3e) and free-logistic (3c) dates:

| Column | Meaning |
|---|---|
| `metric` | start, mid, peak or end |
| `doy`, `date` | the estimated date |
| `ci_low`, `ci_high` | bootstrap 95 % interval (fixed-base logistic only) |
| `qa` | HR-VPP-style quality (curve methods): high / medium / low / filled |
| `at_bound` | fit parameter on its bound |
| `bracket_days` | days between the clear dates just before and after the estimate; more than `GAP_DAYS` = the date lies inside a data gap |
| `trusted` | `bracket_days` ≤ `GAP_DAYS`, `qa` not low / filled and not `at_bound` |
| `preliminary` | year in `INCOMPLETE` |

In [ ]:
rows = []
for (zone, year), r in eos_table_all.iterrows():
    rows.append({"index": "NDVI", "zone": zone, "year": year, "method": "MA midpoint", "metric": "mid", "doy": r.eos_mid_doy})
    rows.append({"index": "NDVI", "zone": zone, "year": year, "method": "MA fixed threshold", "metric": "end",
                 "doy": r.eos_fixed_doy})
for (name, zone, year), r in logistic_table_all.iterrows():
    if pd.isna(r.mid_doy):
        continue
    o = obs_by_zone[name][zone]
    doy = o[(o.year == year) & o.doy.between(*FIT_WINDOW)].doy.values
    qa = hrvpp_qa(doy, r.onset_doy, r.end_doy)
    rows += [{"index": name, "zone": zone, "year": year, "method": "logistic, free", "metric": m, "doy": r[f"{c}_doy"],
              "qa": qa, "at_bound": bool(r.at_bound), "n_obs": r.n_obs}
             for m, c in [("start", "onset"), ("mid", "mid"), ("end", "end")]]

season_dates = pd.DataFrame(rows + fit_rows + whittaker_rows)
season_dates = season_dates[season_dates.doy.notna()].reset_index(drop=True)


def bracket_days(group):
    # Days between the last clear date before and the first after each estimate (inf if none on a side)
    name, zone, year = group.name
    o = obs_by_zone[name][zone]
    doy = np.concatenate([[-np.inf], np.sort(o[(o.year == year) & (o.w > 0)].doy.values.astype(float)), [np.inf]])
    i = np.searchsorted(doy, group.doy.values, side="right")
    return pd.Series(doy[i] - doy[i - 1], index=group.index)


season_dates["bracket_days"] = season_dates.groupby(["index", "zone", "year"], group_keys=False)[["doy"]].apply(
    bracket_days)
season_dates["trusted"] = ((season_dates.bracket_days <= GAP_DAYS) & ~season_dates.qa.isin(["low", "filled"])
                           & ~season_dates.at_bound.eq(True))
season_dates["preliminary"] = season_dates.year.isin(INCOMPLETE)
season_dates["date"] = pd.to_datetime(season_dates.year.astype(str)) + pd.to_timedelta(season_dates.doy.round() - 1, unit="D")
season_dates = season_dates.sort_values(["metric", "method", "index", "zone", "year"]).reset_index(drop=True)
print(f"{len(season_dates)} dates in season_dates")

### 3h. The fitted curves (whole AOI)

One row per index, one column per year (`*` = preliminary): clear dates (× = weight 0: snow or outlier), the post-season level (dotted) and the curves of every method.

In [ ]:
CURVE_DOY = np.arange(SEASON_WINDOW[0], SEASON_WINDOW[1] + 1)
AUTUMN_DOY = CURVE_DOY[CURVE_DOY >= FIT_WINDOW[0]]

fig, axes = plt.subplots(len(INDICES), len(years), figsize=(2.3 * len(years), 2.3 * len(INDICES) + 0.8),
                         sharex=True, sharey="row", squeeze=False)
for i, name in enumerate(INDICES):
    o_all, moving, wc = obs_by_zone[name]["all"], smooth(clean_by_zone[name]["all"]), wcurves[name]["all"]
    for j, year in enumerate(years):
        ax = axes[i, j]
        o = o_all[o_all.year == year]
        ax.scatter(o[o.w > 0].doy, o[o.w > 0].y, s=6, color=INDEX_COLORS[name])
        ax.scatter(o[o.w == 0].doy, o[o.w == 0].y, s=12, marker="x", color=INDEX_COLORS[name])
        m, z = moving[moving.index.year == year], wc[wc.index.year == year]
        ax.plot(m.index.dayofyear, m, color=METHOD_COLORS["MA midpoint"], lw=0.8)
        ax.plot(z.index.dayofyear, z, color=METHOD_COLORS["Whittaker"], lw=1.2)
        if (name, "all", year) in fixed_fits:
            ax.plot(AUTUMN_DOY, logistic_base(AUTUMN_DOY, *fixed_fits[(name, "all", year)]),
                    color=METHOD_COLORS["logistic, fixed base"], lw=1)
        if (name, "all", year) in dl_fits:
            ax.plot(CURVE_DOY, double_logistic(CURVE_DOY, *dl_fits[(name, "all", year)]),
                    color=METHOD_COLORS["double logistic"], lw=1)
        if (name, year) in fits and fits[(name, year)][2] is not None:
            ax.plot(AUTUMN_DOY, logistic(AUTUMN_DOY, *fits[(name, year)][2]), color=METHOD_COLORS["logistic, free"],
                    lw=1, ls="--")
        ax.axhline(base[name].level["all"], color="grey", ls=":", lw=0.8)
        ax.grid(alpha=0.3)
        if i == 0:
            ax.set_title(f"{year}*" if year in INCOMPLETE else str(year), fontsize=9)
        if i == len(INDICES) - 1:
            ax.set_xlabel("DOY")
    axes[i, 0].set_ylabel(name)
    axes[i, 0].set_ylim(*clear_limits(name))
axes[0, 0].set_xlim(*SEASON_WINDOW)
handles = [plt.Line2D([], [], color=METHOD_COLORS[m], lw=1.5, ls="--" if m == "logistic, free" else "-", label=label)
           for m, label in [("Whittaker", "Whittaker"), ("logistic, fixed base", "logistic, fixed base"),
                            ("double logistic", "double logistic"), ("logistic, free", "logistic, free"),
                            ("MA midpoint", "moving average")]]
fig.legend(handles=handles, loc="lower center", ncol=len(handles), fontsize=8)
fig.suptitle("Curves fitted per year (whole AOI)")
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

## 4. Start of the yellowing season: every method

**Start** = the needles begin to lose chlorophyll: greenness (GRVI, GCC, NDVI) leaves its summer level. Every method gives a start for every index; for the rising indices (PSRI, RCC, NDYI), "amplitude left" is the part of the rise still to come.

| Method | Start = |
|---|---|
| `Whittaker` | last day before the 50 % point with ≥ 90 % of the amplitude left (MCD12Q2 "senescence"; Gray et al. 2019) |
| `logistic, fixed base`, `double logistic` | 90 % of the amplitude left: `t0 + s · ln((1 − f) / f)` with `f` = 0.9 |
| `double logistic, 3rd derivative` | first zero of the third derivative, `A − 1.317 / mA` (about 79 % left; Busetto et al. 2010) |
| `logistic, free` | 80 % left (`TRANSITION_FRACTIONS` onset) |

There is no independent reference for the start; section 7 checks it only for its order (before the peak) and data support.

**How sections 4–6 show the methods:**
- **Table:** the whole-AOI day of year per method, index and year (`*` = preliminary), and `trusted_share`, the share of the zone dates that are `trusted` (3g).
- **Figure:** one panel per method with the whole-AOI dates per year, one line per index; hollow = not trusted. Where they exist, the MeteoSwiss station dates are drawn in black. The stations lie lower than the AOI, so compare the year-to-year ups and downs, not the level.

In [ ]:
TARGET_METRICS = {"start": ["start"], "peak": ["peak", "mid"], "end": ["end"]}   # metrics that are candidates
VALIDATE_WITH = {"peak": "mlard94d", "end": "mlard95d"}      # MeteoSwiss phase per date type
complete_years = [y for y in years if y not in INCOMPLETE]

# ---- validation data from 01b_download_validation (each part is skipped if its file is missing)
pheno, hrvpp_end = None, None
if (VALIDATION_DIR / "meteoswiss_larch.csv").exists():
    p = pd.read_csv(VALIDATION_DIR / "meteoswiss_larch.csv")
    p = p[p.year.isin(complete_years)]
    p["anomaly"] = p.doy - p.groupby(["station", "param"]).doy.transform("mean")   # each station vs its own mean
    pheno = p.groupby(["param", "year"]).agg(anomaly=("anomaly", "mean"), doy=("doy", "mean"),
                                            n_stations=("station", "nunique"), height=("height_masl", "mean"))
else:
    print("No MeteoSwiss observations (run 01b_download_validation)")
if (VALIDATION_DIR / "hrvpp_by_zone.csv").exists():
    h = pd.read_csv(VALIDATION_DIR / "hrvpp_by_zone.csv")
    hrvpp_end = h[h["product"] == "EOSD"].set_index(["zone", "year"]).doy_median.dropna()
if hrvpp_end is None or hrvpp_end.empty:
    hrvpp_end = None
    print("No HR-VPP end-of-season dates (needs a WEkEO login, see 01b_download_validation)")


def reference_dates(target):
    # [(label, line style, DOY per year)] of the validation data of one date type (whole AOI)
    refs = []
    if pheno is not None and target in VALIDATE_WITH:
        st = pheno.xs(VALIDATE_WITH[target], level="param")
        refs.append((f"MeteoSwiss {PHENO_PARAMS[VALIDATE_WITH[target]]} (stations, mean {st.height.mean():.0f} m)",
                     "ks--", st.doy))
    if hrvpp_end is not None and target == "end" and "all" in hrvpp_end.index.get_level_values("zone"):
        refs.append(("HR-VPP end of season", "kD-", hrvpp_end.xs("all", level="zone")))
    return refs


def show_candidates(target):
    # Table and figure of every method of one date type (whole AOI)
    keys = ["method", "index", "metric"]
    d = season_dates[season_dates.metric.isin(TARGET_METRICS[target])]
    aoi = d[d.zone == "all"]
    table = aoi.pivot_table(index=keys, columns="year", values="doy").round(0)
    table.columns = [f"{y}*" if y in INCOMPLETE else str(y) for y in table.columns]
    zones = d[(d.zone != "all") & ~d.preliminary]
    table["trusted_share"] = zones.groupby(keys).trusted.mean().round(2)
    display(table)

    methods = list(dict.fromkeys(table.index.get_level_values("method")))
    ncol = min(len(methods), 3)
    nrow = int(np.ceil(len(methods) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 3.4 * nrow + 0.8), sharex=True, sharey=True,
                             squeeze=False)
    for ax, method in zip(axes.flat, methods):
        for (index, metric), g in aoi[aoi.method == method].groupby(["index", "metric"]):
            g = g.sort_values("year")
            color, marker = INDEX_COLORS[index], INDEX_MARKERS.get(index, "o")
            ls = "--" if metric == "mid" and target == "peak" else "-"
            ax.plot(g.year, g.doy, color=color, ls=ls, lw=1)
            for trusted, mfc in [(True, color), (False, "white")]:
                t = g[g.trusted == trusted]
                ax.plot(t.year, t.doy, ls="", marker=marker, color=color, mfc=mfc, ms=6)
        for label, style, s in reference_dates(target):
            ax.plot(s.index, s.values, style, lw=1.5, ms=5)
        for year in INCOMPLETE:
            ax.axvspan(year - 0.5, year + 0.5, color="grey", alpha=0.1, lw=0)
        ax.set_title(method)
        ax.xaxis.get_major_locator().set_params(integer=True)
        ax.grid(alpha=0.3)
    for ax in axes.flat[len(methods):]:
        ax.set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("day of year")
    line = lambda **kw: plt.Line2D([], [], **kw)
    handles = [line(color=INDEX_COLORS[i], marker=INDEX_MARKERS.get(i, "o"), label=i) for i in dict.fromkeys(aoi["index"])]
    handles.append(line(color="grey", marker="o", mfc="white", ls="", label="not trusted"))
    if target == "peak":
        handles.append(line(color="grey", ls="--", label="mid (50 %) as peak"))
    handles += [line(color="black", ls=style[2:], marker=style[1], label=label) for label, style, _ in reference_dates(target)]
    fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 6), fontsize=8)
    fig.suptitle(f"{target.capitalize()}: whole-AOI dates per year (shaded: preliminary)")
    plt.tight_layout(rect=(0, 0.06 if len(handles) <= 6 else 0.1, 1, 1))
    plt.show()


show_candidates("start")

## 5. Peak of the yellowing: every method

**Peak** = the canopy is most yellow: chlorophyll gone, needles still on. Two kinds of candidates:

| Method | Peak = |
|---|---|
| `Whittaker`, yellowness index, `peak` | day of the curve's maximum in `PEAK_WINDOW` (PSRI; RCC and NDYI once downloaded) |
| every curve method, `mid` | 50 % of the amplitude left: the MCD12Q2 "greendown midpoint" of a greenness decline (dashed in the figure) |
| `MA midpoint` (NDVI) | the moving average crosses halfway between its yearly maximum and the lowest value after it (baseline, 3e) |

Reference: MeteoSwiss needle colouring 50 % (`mlard94d`).

In [ ]:
show_candidates("peak")

## 6. End of the season: every method

**End** = the needles have fallen: NDVI reaches its post-season level.

| Method | End = |
|---|---|
| `Whittaker` | first day after the 50 % point with ≤ 15 % of the amplitude left (HR-VPP end of season; Tian et al. 2021) |
| `logistic, fixed base`, `double logistic` | 15 % of the amplitude left |
| `double logistic, 3rd derivative` | second zero of the third derivative, `A + 1.317 / mA` (about 21 % left) |
| `logistic, free` | 20 % left (`TRANSITION_FRACTIONS` end) |
| `MA fixed threshold` (NDVI) | the moving average falls below `EOS_FIXED_THRESHOLD` (baseline, 3e) |

References: MeteoSwiss needle drop 50 % (`mlard95d`) and the HR-VPP end of season (if downloaded in `01b`). There are few clear dates after mid-October, so many end dates lie in a data gap (low `trusted_share`).

In [ ]:
show_candidates("end")

## 7. Comparison of all methods

One table over every candidate (method × index × date type) of sections 4–6. Peak candidates are the yellowness maxima (`peak`) and the 50 % points of the declines (`mid`, the MCD12Q2 "greendown midpoint"). Complete years only.

| Column | Meaning |
|---|---|
| `coverage` | share of zone-years with a date |
| `observed` | share of dates with a clear date within `GAP_DAYS` on both sides (not inside a data gap) |
| `qa_good` | share with QA high or medium (curve methods) |
| `ci_days` | median width of the bootstrap 95 % interval (fixed-base logistic) |
| `order_ok` | share of zone-years in the order start < peak < end, judged against the median of all candidates of the neighbouring date types |
| `days_per_100m` | within-year elevation gradient across zones; negative = earlier higher up, as expected (elevation controls larch phenology; Migliavacca et al. 2008) |
| `aoi_sd_years` | year-to-year standard deviation of the whole-AOI date |
| `ms_r`, `ms_rmse`, `ms_n` | whole-AOI anomalies vs MeteoSwiss station anomalies: needle colouring 50 % for the peak, needle drop 50 % for the end. Correlation, RMSE in days, number of years |
| `ms_offset` | mean AOI date − mean station date. The stations lie lower (1050–1930 m), so this is not an error |
| `hrvpp_mae`, `hrvpp_bias`, `hrvpp_r` | end dates vs HR-VPP end of season per zone-year (if downloaded in `01b`) |

**Short list:** `mean_rank` is the mean of a candidate's ranks within its date type over `ms_r`, `hrvpp_mae`, `observed`, `order_ok` and `qa_good` (where available; rank 1 = best). The `AVERAGE_TOP_N` best candidates with `coverage` ≥ 0.5 form the short list that `"average"` uses in 7b. This is a heuristic ranking, not a final choice.

**Figure:** the whole-AOI mean date ± sd over the years of every candidate with `coverage` ≥ 0.5, best rank on top. Filled = on the short list; vertical lines = mean date of the reference data. With about 9 years, a correlation needs |r| > 0.67 to be significant at 5 %.

In [ ]:
d = season_dates[~season_dates.preliminary].dropna(subset=["doy"])
complete_years = [y for y in sorted(set(stats_all.index.year)) if y not in INCOMPLETE]
n_zone_years = (len(ZONES) - 1) * len(complete_years)
# ---- candidates and the consensus (median of all candidates) per zone-year and date type
candidates = {(t, *k): g.set_index(["zone", "year"]).doy for t, metrics in TARGET_METRICS.items()
              for k, g in d[d.metric.isin(metrics)].groupby(["method", "index", "metric"])}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # zone-years without any candidate of a date type
    consensus = pd.DataFrame({t: pd.concat([s for k, s in candidates.items() if k[0] == t], axis=1).median(axis=1)
                              for t in TARGET_METRICS})


def order_share(target, s):
    c = consensus.reindex(s.index)
    ok = {"start": s < c["peak"], "peak": (c["start"] < s) & (s < c["end"]), "end": s > c["peak"]}[target]
    known = {"start": c["peak"].notna(), "peak": c["start"].notna() & c["end"].notna(), "end": c["peak"].notna()}[target]
    return ok[known].mean() if known.any() else np.nan


def elevation_slope(s):
    # Days per 100 m from zone dates, within years (each year's mean removed from date and elevation)
    z = s.drop(index="all", level="zone", errors="ignore").reset_index()
    z["x"] = z.zone.map(zone_table.elev_mean) / 100
    z = z[z.groupby("year").x.transform("size") >= 2]
    x, y = z.x - z.groupby("year").x.transform("mean"), z.doy - z.groupby("year").doy.transform("mean")
    return (x @ y) / (x @ x) if len(z) > 3 and (x @ x) > 0 else np.nan


def meteoswiss_check(target, aoi):
    if pheno is None or target not in VALIDATE_WITH:
        return {}
    st = pheno.xs(VALIDATE_WITH[target], level="param")
    common = aoi.index.intersection(st.index)
    if len(common) < 4:
        return {"ms_n": len(common)}
    a = aoi[common] - aoi[common].mean()
    s = st.anomaly[common] - st.anomaly[common].mean()
    return {"ms_r": a.corr(s), "ms_rmse": np.sqrt(((a - s) ** 2).mean()), "ms_n": len(common),
            "ms_offset": (aoi[common] - st.doy[common]).mean()}


def hrvpp_check(target, s):
    if hrvpp_end is None or target != "end":
        return {}
    m = pd.concat({"doy": s, "ref": hrvpp_end}, axis=1, join="inner").dropna()
    if len(m) < 4:
        return {"hrvpp_n": len(m)}
    e = m.doy - m.ref
    return {"hrvpp_mae": e.abs().mean(), "hrvpp_bias": e.mean(), "hrvpp_r": m.doy.corr(m.ref), "hrvpp_n": len(m)}


rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # medians / means of candidates without any value
    for (target, method, index, metric), s in candidates.items():
        g = d[(d.method == method) & (d["index"] == index) & (d.metric == metric)]
        gz = g[g.zone != "all"]
        aoi = s.xs("all", level="zone") if "all" in s.index.get_level_values("zone") else pd.Series(dtype=float)
        rows.append({"target": target, "method": method, "index": index, "metric": metric,
                     "coverage": len(gz) / n_zone_years, "observed": (gz.bracket_days <= GAP_DAYS).mean(),
                     "qa_good": gz.qa.isin(["high", "medium"]).mean() if gz.qa.notna().any() else np.nan,
                     "ci_days": (gz.ci_high - gz.ci_low).median() if "ci_high" in gz else np.nan,
                     "order_ok": order_share(target, s.drop(index="all", level="zone", errors="ignore")),
                     "days_per_100m": elevation_slope(s), "aoi_mean_doy": aoi.mean(), "aoi_sd_years": aoi.std(),
                     **meteoswiss_check(target, aoi), **hrvpp_check(target, s)})
comparison = pd.DataFrame(rows).set_index(["target", "method", "index", "metric"]).sort_index()

# ---- short list per date type: mean rank over the criteria a candidate has (rank 1 = best within its date type;
#      only candidates with coverage >= 0.5 are ranked)
CRITERIA = {"ms_r": False, "hrvpp_mae": True, "observed": False, "order_ok": False, "qa_good": False}   # ascending?
CRITERIA = {c: asc for c, asc in CRITERIA.items() if c in comparison and comparison[c].notna().any()}
ranked = comparison[comparison.coverage >= 0.5]
ranks = pd.DataFrame({c: ranked.groupby(level="target")[c].rank(ascending=asc) for c, asc in CRITERIA.items()})
comparison["mean_rank"] = ranks.mean(axis=1)
display(comparison.round(2))
print(f"Short list (coverage >= 0.5; mean rank over {', '.join(CRITERIA)}):")
shortlist = {}
for target in TARGET_METRICS:
    t = comparison.xs(target, level="target")
    t = t[t.coverage >= 0.5].sort_values("mean_rank")
    shortlist[target] = list(t.index[:AVERAGE_TOP_N] if AVERAGE_TOP_N else t.index)
    for (method, index, metric), r in t.head(len(shortlist[target])).iterrows():
        ms = f", MeteoSwiss r = {r.ms_r:.2f}" if "ms_r" in r and pd.notna(r.ms_r) else ""
        print(f"  {target:5s}  {method} / {index} / {metric}: mean rank {r.mean_rank:.1f}, observed {r.observed:.0%}, "
              f"order {r.order_ok:.0%}{ms}")

# ---- figure: mean date +- sd over the years per candidate (whole AOI), best rank on top
n_max = max(len(ranked.xs(t, level="target")) for t in TARGET_METRICS)
fig, axes = plt.subplots(1, len(TARGET_METRICS), figsize=(6.5 * len(TARGET_METRICS), 0.32 * n_max + 2), squeeze=False)
for ax, target in zip(axes[0], TARGET_METRICS):
    t = comparison.loc[ranked.index].xs(target, level="target").sort_values("mean_rank", ascending=False)
    for k, (key, r) in enumerate(t.iterrows()):
        method, index, metric = key
        color = METHOD_COLORS[method]
        ax.errorbar(r.aoi_mean_doy, k, xerr=r.aoi_sd_years, fmt=INDEX_MARKERS.get(index, "o"), color=color,
                    mfc=color if key in shortlist[target] else "white", ms=7, capsize=3)
    for (label, style, s), ls in zip(reference_dates(target), ["--", ":"]):
        ax.axvline(s.mean(), color="black", ls=ls, lw=1.2, label=label)
    ax.set_yticks(range(len(t)), [" / ".join(key) for key in t.index], fontsize=8)
    ax.set_title(f"{target}: mean ± sd over years (whole AOI)")
    ax.set_xlabel("day of year")
    ax.grid(alpha=0.3)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=7, loc="lower left")
handles = ([plt.Line2D([], [], color=METHOD_COLORS[m], lw=2, label=m) for m in dict.fromkeys(ranked.reset_index().method)]
           + [plt.Line2D([], [], color="grey", marker="o", ls="", label="on the short list"),
              plt.Line2D([], [], color="grey", marker="o", mfc="white", ls="", label="not on the short list")])
fig.legend(handles=handles, loc="lower center", ncol=len(handles), fontsize=8)
plt.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

### 7b. Chosen dates

`SEASON_CHOICE` in `config.py` sets the date used in sections 8 and 9, per date type:
- `"average"`: the mean of the short list (the `AVERAGE_TOP_N` best candidates; `None` = every candidate with `coverage` ≥ 0.5). Each candidate enters as its anomaly from its own mean over all zone-years, plus the mean level of all chosen candidates, so a candidate missing in one year does not shift that year's level.
- one candidate, written as in the table above, e.g. `"Whittaker / GRVI / mid"`.

`selected` holds the chosen start / peak / end per zone and year (complete years). The figure shows the season of the whole AOI per year with the reference data.

In [ ]:
def chosen_keys(target):
    choice = SEASON_CHOICE[target]
    if choice == "average":
        if not shortlist[target]:
            raise ValueError(f"No {target} candidate with coverage >= 0.5; set SEASON_CHOICE[{target!r}] to one candidate")
        return shortlist[target]
    key = tuple(part.strip() for part in choice.split("/"))
    if (target, *key) not in candidates:
        options = [" / ".join(k[1:]) for k in candidates if k[0] == target]
        raise ValueError(f"SEASON_CHOICE[{target!r}] = {choice!r} is not a {target} candidate. Options: {options}")
    return [key]


def combine(target, keys):
    # Mean of the chosen candidates per zone-year: each candidate's anomaly from its own mean over all zone-years,
    # averaged over the candidates present, plus the mean level of all of them
    s = pd.concat([candidates[(target, *k)] for k in keys], axis=1, ignore_index=True)
    return s.sub(s.mean()).mean(axis=1) + s.mean().mean()


def doy_to_date(year, doy):
    return date(int(year), 1, 1) + timedelta(days=int(round(doy)) - 1) if pd.notna(doy) else None


chosen = {target: chosen_keys(target) for target in TARGET_METRICS}
choice_label = {t: f"average of {len(k)}" if len(k) > 1 else " / ".join(k[0]) for t, k in chosen.items()}
for target, keys in chosen.items():
    print(f"{target}: {choice_label[target]}: {'; '.join(' / '.join(k) for k in keys)}")
selected = pd.DataFrame({target: combine(target, keys) for target, keys in chosen.items()}).sort_index()

z = selected.drop(index="all", level="zone").dropna()
print(f"start < peak < end in {((z.start < z.peak) & (z.peak < z.end)).mean():.0%} of {len(z)} zone-years")
aoi_sel = selected.xs("all", level="zone")
display(aoi_sel.round(0).join(pd.DataFrame({f"{t}_date": [doy_to_date(y, v) for y, v in aoi_sel[t].items()]
                                            for t in TARGET_METRICS}, index=aoi_sel.index)))

MONTH_TICKS = {date(2023, m, 1).timetuple().tm_yday: f"1 {date(2023, m, 1):%b}" for m in range(8, 13)}
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.vlines(aoi_sel.index, aoi_sel.start, aoi_sel.end, color="tab:olive", lw=10, alpha=0.35, label="start to end")
ax.plot(aoi_sel.index, aoi_sel.start, "_", color="tab:olive", ms=18, mew=2, label=f"start ({choice_label['start']})")
ax.plot(aoi_sel.index, aoi_sel.peak, "o", color="darkorange", ms=9, label=f"peak ({choice_label['peak']})")
ax.plot(aoi_sel.index, aoi_sel.end, "_", color="saddlebrown", ms=18, mew=2, label=f"end ({choice_label['end']})")
for target, marker in [("peak", "s"), ("end", "D")]:
    for label, _, s in reference_dates(target):
        ax.plot(s.index + 0.2, s.values, marker, color="black", mfc="white", ms=6, ls="", label=label)
ax.set_yticks(list(MONTH_TICKS), [f"{v} ({k})" for k, v in MONTH_TICKS.items()])
ax.set_xlabel("year")
ax.xaxis.get_major_locator().set_params(integer=True)
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
ax.set_title("Chosen season dates per year (whole AOI; tick labels: date (DOY))")
plt.tight_layout()
plt.show()

## 8. Trend and short-term projection (no climate drivers)

For each chosen date (7b) of the whole AOI and the complete years:
- **OLS trend** of the day of year on the year, with its 95 % confidence interval, extrapolated `FORECAST_YEARS` ahead with a 95 % prediction interval (the range a single future year is expected to fall in).
- **Mean model** (no trend) alongside. If the trend is not significant (`trend_significant` = False), the mean ± interval is the more honest expectation.

With about 9 years, one early or late year at either end moves the slope a lot. This is the statistical baseline that the climate models in `04` are compared against.

In [ ]:
proj_rows = []
level = f"{1 - ALPHA:.0%}"
fig, axes = plt.subplots(1, len(TARGET_METRICS), figsize=(5.5 * len(TARGET_METRICS), 4.5), squeeze=False)
for ax, target in zip(axes[0], TARGET_METRICS):
    d = aoi_sel[target].dropna()
    n = len(d)
    if n < 4:
        ax.set_title(f"{target}: only {n} years, skipped")
        continue
    years_fit = d.index.values.astype(float)
    y = d.values.astype(float)
    x0 = years_fit.mean()   # centred years for numerical stability
    ols = sm.OLS(y, sm.add_constant(years_fit - x0)).fit()
    mean_model = sm.OLS(y, np.ones((n, 1))).fit()
    future = np.arange(years_fit.max() + 1, years_fit.max() + 1 + FORECAST_YEARS)
    grid = np.arange(years_fit.min(), future.max() + 1)
    pred = ols.get_prediction(sm.add_constant(grid - x0, has_constant="add")).summary_frame(alpha=ALPHA)
    pred_mean = mean_model.get_prediction(np.ones((len(grid), 1))).summary_frame(alpha=ALPHA)
    slope_lo, slope_hi = ols.conf_int(ALPHA)[1]
    significant = bool(ols.pvalues[1] < ALPHA)
    print(f"{target} ({choice_label[target]}), {n} years: OLS slope {ols.params[1]:+.2f} d/yr "
          f"({level} CI {slope_lo:+.2f} to {slope_hi:+.2f}), p = {ols.pvalues[1]:.3f}, R2 = {ols.rsquared:.2f}")
    for i, yr in enumerate(grid):
        if yr in future:
            proj_rows.append({
                "date_type": target, "year": int(yr),
                "trend_doy": pred["mean"].iloc[i],
                "trend_pi_low": pred["obs_ci_lower"].iloc[i], "trend_pi_high": pred["obs_ci_upper"].iloc[i],
                "mean_doy": pred_mean["mean"].iloc[i],
                "mean_pi_low": pred_mean["obs_ci_lower"].iloc[i], "mean_pi_high": pred_mean["obs_ci_upper"].iloc[i],
                "trend_significant": significant,
            })

    color = "tab:blue"
    ax.fill_between(grid, pred["obs_ci_lower"], pred["obs_ci_upper"], color=color, alpha=0.15,
                    label=f"OLS {level} prediction interval")
    ax.fill_between(grid, pred["mean_ci_lower"], pred["mean_ci_upper"], color=color, alpha=0.3,
                    label=f"OLS {level} CI of the trend")
    ax.plot(grid, pred["mean"], color=color, lw=2, label="OLS trend")
    ax.plot(grid, pred_mean["mean"], "--", color="grey", lw=1, label="mean (no trend)")
    ax.fill_between(grid, pred_mean["obs_ci_lower"], pred_mean["obs_ci_upper"], facecolor="none",
                    edgecolor="grey", hatch="//", lw=0, alpha=0.5, label=f"mean {level} prediction interval")
    ax.plot(years_fit, y, "o", color="black", label="chosen date")
    ax.axvspan(future.min() - 0.5, future.max() + 0.5, color="grey", alpha=0.08)
    ax.set_title(f"{target} ({choice_label[target]}): {ols.params[1]:+.2f} d/yr, p = {ols.pvalues[1]:.2f}")
    ax.set_xlabel("year")
    ax.xaxis.get_major_locator().set_params(integer=True)
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("day of year")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(handles), fontsize=8)
plt.suptitle("Trend and projection of the chosen dates (whole AOI; shaded columns = projected years)")
plt.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

season_projection = pd.DataFrame(proj_rows)
if not season_projection.empty:
    season_projection = season_projection.set_index(["date_type", "year"]).round(1)
    for c in ["trend_doy", "mean_doy"]:
        season_projection[c.replace("_doy", "_date")] = [doy_to_date(yr, v) for (_, yr), v in season_projection[c].items()]
    display(season_projection)

## 9. Phenology vs topography (elevation band × aspect)

Uses the chosen dates of 7b per zone and year.

- **Map:** elevation band and aspect class of the AOI pixels (`zones.tif` from `01`) on a hillshade. The hillshade is lit from the south, so north-facing slopes appear dark: blue (N) pixels should sit on the dark side of the valleys.
- **Table:** zone attributes, the mean number of clear autumn dates per year and index (`autumn_dates_*`, `FIT_WINDOW[0]`–`POST_WINDOW[1]`) and the mean share of SCL "dark" pixels in autumn.
- **Plot:** day of year against elevation band, one line per aspect (mean ± sd over years).
- **Model:** OLS `doy ~ elev_100m + C(aspect) + C(year)`.
  - The elevation coefficient is in days per 100 m.
  - The aspect coefficient is the north − south difference in days.
  - `C(year)` absorbs year-to-year weather.

**Caveats:**
- Zones share weather and clouds within a year, so the CIs are somewhat optimistic.
- North slopes lose more autumn dates to SCL class 2 (dark / topographic shadow); see `autumn_dates_*` and `autumn_dark_fraction`.
- PSRI on north slopes in late autumn (low sun) is the most affected.

In [ ]:
zr = rioxarray.open_rasterio(CACHE_DIR / "zones.tif", masked=True)
elev, elev_band, aspect = (zr.sel(band=b).values for b in (1, 2, 3))
on = ~np.isnan(elev_band)
yy, xx = np.meshgrid(zr.y.values, zr.x.values, indexing="ij")
left, bottom, right, top = zr.rio.bounds()
shade = LightSource(azdeg=180, altdeg=30).hillshade(np.nan_to_num(elev, nan=np.nanmean(elev)),
                                                   dx=RESOLUTION, dy=RESOLUTION)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax in axes:
    ax.imshow(shade, cmap="gray", extent=(left, right, bottom, top), alpha=0.6)
bands = np.unique(elev_band[on])
for b, color in zip(bands, plt.get_cmap("viridis")(np.linspace(0, 1, len(bands)))):
    sel = on & (elev_band == b)
    axes[0].scatter(xx[sel], yy[sel], s=0.2, color=color, label=f"{int(b)}-{int(b + ELEV_STEP)} m")
for asp, code in ASPECT_CODES.items():
    sel = on & (aspect == code)
    axes[1].scatter(xx[sel], yy[sel], s=0.2, color=ASPECT_COLORS[asp], label=f"{asp} ({int(sel.sum())} px)")
for ax, title in zip(axes, ["Elevation band of the AOI pixels", "Aspect class (hillshade lit from the south)"]):
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.legend(markerscale=20, fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
zone_attrs = zone_table.drop(index="all")[["elev_band", "aspect", "elev_mean"]]
autumn = stats_all[stats_all.index.dayofyear >= FIT_WINDOW[0]]
autumn_dates = pd.DataFrame({
    f"autumn_dates_{name}": {zone: o[(o.w > 0) & o.doy.between(FIT_WINDOW[0], POST_WINDOW[1]) & ~o.year.isin(INCOMPLETE)]
                             .groupby("year").size().mean() for zone, o in obs_by_zone[name].items()}
    for name in INDICES})
coverage = (zone_table[["elev_band", "aspect", "n_pixels", "elev_mean", "slope_mean", "northness_mean"]]
            .join(autumn_dates)
            .join(autumn.groupby("zone").scl_dark_fraction.mean().rename("autumn_dark_fraction")))
display(coverage.round(2))

zone_dates = selected.drop(index="all", level="zone").reset_index().join(zone_attrs, on="zone", how="inner")

fig, axes = plt.subplots(1, len(TARGET_METRICS), figsize=(4.5 * len(TARGET_METRICS), 4), sharex=True, squeeze=False)
for ax, target in zip(axes[0], TARGET_METRICS):
    for asp, color in ASPECT_COLORS.items():
        d = zone_dates[zone_dates.aspect == asp].dropna(subset=[target])
        if d.empty:
            continue
        shift = ELEV_STEP / 2 + {"N": -0.08, "S": 0.08, "flat": 0}[asp] * ELEV_STEP   # band centre, dodged
        ax.scatter(d.elev_band + shift, d[target], s=8, color=color, alpha=0.3)
        g = d.groupby("elev_band")[target].agg(["mean", "std"])
        ax.errorbar(g.index + shift, g["mean"], yerr=g["std"].fillna(0), fmt="o-", color=color, capsize=3,
                    label=asp)
    ax.set_title(f"{target} ({choice_label[target]})")
    ax.set_xlabel("elevation band centre (m)")
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("day of year")
axes[0, 0].legend(title="aspect")
plt.suptitle("Chosen dates vs elevation and aspect (points: zone-years, lines: mean ± sd over years)")
plt.tight_layout()
plt.show()

NS_TERM = "C(aspect, Treatment('S'))[T.N]"


def topo_model(data, col):
    d = data.dropna(subset=[col]).assign(elev_100m=lambda x: x.elev_mean / 100)
    if (not {"N", "S"} <= set(d.aspect) or d.elev_band.nunique() < 2
            or len(d) < d.year.nunique() + d.aspect.nunique() + 3):
        return None
    return smf.ols(f"{col} ~ elev_100m + C(aspect, Treatment('S')) + C(year)", data=d).fit()
rows = []
for target in TARGET_METRICS:
    fit = topo_model(zone_dates, target)
    if fit is None:
        print(f"{target}: too few zone-years (needs N and S zones and >= 2 elevation bands), no model")
        continue
    ci = fit.conf_int(ALPHA)
    rows.append({
        "date_type": target, "method": choice_label[target], "n_zone_years": int(fit.nobs),
        "n_zones": zone_dates.dropna(subset=[target]).zone.nunique(),
        "days_per_100m": fit.params["elev_100m"],
        "elev_ci_low": ci.loc["elev_100m", 0], "elev_ci_high": ci.loc["elev_100m", 1],
        "elev_p": fit.pvalues["elev_100m"],
        "north_minus_south_days": fit.params[NS_TERM],
        "ns_ci_low": ci.loc[NS_TERM, 0], "ns_ci_high": ci.loc[NS_TERM, 1], "ns_p": fit.pvalues[NS_TERM],
        "r2": fit.rsquared,
    })
topo_effects = pd.DataFrame(rows).set_index("date_type").round(3) if rows else pd.DataFrame()
display(topo_effects)

## 10. Trend / seasonal / remainder decomposition (STL, whole AOI)

Robust STL on a regular 5-day grid splits each index into a long-term trend, a repeating yearly cycle and the remainder. It needs at least two full years.

In [ ]:
STEP_DAYS = 5
PERIOD = int(round(365.25 / STEP_DAYS))   # observations per year on the regular grid


def stl_decompose(series, name):
    s = series.resample(f"{STEP_DAYS}D").mean()
    s = s.loc[s.first_valid_index():s.last_valid_index()].interpolate(method="time")
    if len(s) < 2 * PERIOD:
        print(f"{name}: {len(s) * STEP_DAYS / 365.25:.1f} years of data; STL needs at least 2 full years")
        return None
    res = STL(s, period=PERIOD, robust=True).fit()
    return pd.DataFrame({"observed": s, "trend": res.trend, "seasonal": res.seasonal, "remainder": res.resid})


for name in INDICES:
    dec = stl_decompose(medians[name], name)
    if dec is None:
        continue
    color = INDEX_COLORS[name]
    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
    axes[0].plot(dec.index, dec.observed, color="grey", lw=1)
    axes[0].set_ylabel(f"{name} observed")
    axes[1].plot(dec.index, dec.trend, color=color, lw=2)
    axes[1].set_ylabel("trend")
    axes[2].plot(dec.index, dec.seasonal, color=color, lw=1.5)
    axes[2].set_ylabel("seasonal")
    axes[3].scatter(dec.index, dec.remainder, s=6, color=color)
    axes[3].set_ylabel("remainder")
    for a in axes[2:]:
        a.axhline(0, color="black", lw=0.5)
    for a in axes:
        a.grid(alpha=0.3)
    axes[0].set_title(f"{name} STL decomposition (period = {PERIOD} x {STEP_DAYS} days)")
    plt.tight_layout()
    plt.show()
    slope = np.polyfit((dec.index - dec.index[0]).days / 365.25, dec.trend.values, 1)[0]
    print(f"{name} trend slope: {slope:+.4f} per year | seasonal amplitude: "
          f"{dec.seasonal.max() - dec.seasonal.min():.3f} | remainder std: {dec.remainder.std():.3f}")

## 11. Export

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_DIR / "ndvi_timeseries_all_dates.csv")
df_clean.to_csv(OUT_DIR / "ndvi_timeseries_clean.csv")
season_dates.to_csv(OUT_DIR / "season_dates.csv", index=False)
comparison.to_csv(OUT_DIR / "season_method_comparison.csv")
selected.to_csv(OUT_DIR / "season_selected.csv")
season_projection.to_csv(OUT_DIR / "season_projection.csv")
eos_table.to_csv(OUT_DIR / "eos_by_zone.csv")
logistic_table.to_csv(OUT_DIR / "logistic_by_zone.csv")
zone_table.to_csv(OUT_DIR / "zones.csv")
topo_effects.to_csv(OUT_DIR / "topography_effects.csv")
print(f"Results written to {OUT_DIR}")

## References

- Atzberger, C. & Eilers, P. H. C. (2011). A time series for monitoring vegetation activity and phenology at 10-daily time steps covering large parts of South America. *International Journal of Digital Earth* 4(5), 365–386.
- Beck, P. S. A., Atzberger, C., Høgda, K. A., Johansen, B. & Skidmore, A. K. (2006). Improved monitoring of vegetation dynamics at very high latitudes: a new method using MODIS NDVI. *Remote Sensing of Environment* 100(3), 321–334.
- Busetto, L., Colombo, R., Migliavacca, M., et al. (2010). Remote sensing of larch phenological cycle and analysis of relationships with climate in the Alpine region. *Global Change Biology* 16(9), 2504–2517.
- Copernicus Land Monitoring Service (2021). *HR-VPP Algorithm Theoretical Basis Document: Seasonal Trajectories and VPP parameters*, issue 1.1. https://land.copernicus.eu/en/technical-library
- Eilers, P. H. C. (2003). A perfect smoother. *Analytical Chemistry* 75(14), 3631–3636.
- Gray, J., Sulla-Menashe, D. & Friedl, M. A. (2019). *User Guide to Collection 6 MODIS Land Cover Dynamics (MCD12Q2) Product*. NASA LP DAAC.
- Jönsson, P. & Eklundh, L. (2004). TIMESAT – a program for analyzing time-series of satellite sensor data. *Computers & Geosciences* 30(8), 833–845.
- MeteoSwiss. Phenological observations, open data (`ch.meteoschweiz.ogd-phenology`). https://opendatadocs.meteoswiss.ch/a-data-groundbased/a9-phenological-observations
- Merzlyak, M. N., Gitelson, A. A., Chivkunova, O. B. & Rakitin, V. Y. (1999). Non-destructive optical detection of pigment changes during leaf senescence and fruit ripening. *Physiologia Plantarum* 106(1), 135–141.
- Migliavacca, M., Cremonese, E., Colombo, R., et al. (2008). European larch phenology in the Alps: can we grasp the role of ecological factors by combining field observations and inverse modelling? *International Journal of Biometeorology* 52(7), 587–605.
- Motohka, T., Nasahara, K. N., Oguma, H. & Tsuchida, S. (2010). Applicability of green-red vegetation index for remote sensing of vegetation phenology. *Remote Sensing* 2(10), 2369–2387.
- Richardson, A. D., Hufkens, K., Milliman, T., et al. (2018). Tracking vegetation phenology across diverse North American biomes using PhenoCam imagery. *Scientific Data* 5, 180028.
- Sonnentag, O., Hufkens, K., Teshera-Sterne, C., et al. (2012). Digital repeat photography for phenological research in forest ecosystems. *Agricultural and Forest Meteorology* 152, 159–177.
- Sulik, J. J. & Long, D. S. (2016). Spectral considerations for modeling yield of canola. *Remote Sensing of Environment* 184, 161–174.
- Tian, F., Cai, Z., Jin, H., et al. (2021). Calibrating vegetation phenology from Sentinel-2 using eddy covariance, PhenoCam, and PEP725 networks across Europe. *Remote Sensing of Environment* 260, 112456.
- Zeng, L., Wardlow, B. D., Xiang, D., Hu, S. & Li, D. (2020). A review of vegetation phenological metrics extraction using time-series, multispectral satellite data. *Remote Sensing of Environment* 237, 111511.